# Contextual MARL — rliable evaluation notebook

Use this on a **cluster login node** (or locally) after training finishes.

**Workflow**
1. Train with `matd3_pistonball.py` (multiple seeds).
2. Aggregate: `python scripts/rliable_aggregate.py --exp-name matd3_pistonball`
3. Open this notebook and point `REPORT_PATH` at `reports/rliable_<exp>.json`.

**How to read the performance profile**
- X-axis: return threshold τ (team cumulative return on Pistonball).
- Y-axis: fraction of *(seed, context)* pairs with return ≥ τ.
- Higher curves = better and more robust performance.
- Shaded band: 95% stratified bootstrap CI ([rliable](https://github.com/google-research/rliable)).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from evaluation.rliable_utils import build_score_matrix, discover_run_dirs
from evaluation.visualize import (
    aggregate_metrics_table,
    format_aggregate_summary,
    load_report,
    plot_aggregate_intervals,
    plot_generalization_contexts,
    plot_performance_profile,
    plot_sample_efficiency_curve,
)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")
%matplotlib inline

## Configuration
Edit these paths for your cluster run.

In [ ]:
REPO_ROOT = Path(".").resolve()  # project root (where you launch jupyter)
EXP_NAME = "matd3_pistonball"
RUNS_ROOT = REPO_ROOT / "runs"
REPORT_PATH = REPO_ROOT / "reports" / f"rliable_{EXP_NAME}.json"

# Optional: compare a second method (context concat vs hidden baseline)
COMPARE_REPORT_PATH = None  # e.g. REPO_ROOT / "reports" / "rliable_matd3_context_concat.json"

## Load report
If the JSON is missing, run the aggregate script first (see top of notebook).

In [ ]:
if not REPORT_PATH.exists():
    raise FileNotFoundError(
        f"Report not found: {REPORT_PATH}\n"
        f"Run: python scripts/rliable_aggregate.py --exp-name {EXP_NAME}"
    )

report = load_report(REPORT_PATH)
run_dirs = discover_run_dirs(RUNS_ROOT, EXP_NAME, seeds=report.get("seeds"))
score_matrix, context_order, seeds = build_score_matrix(run_dirs)

print(format_aggregate_summary(report))
print(f"\nScore matrix shape: {score_matrix.shape} (seeds × contexts)")

## Aggregate metrics (table)
Prefer **IQM** over plain mean for small numbers of seeds (rliable recommendation).

In [ ]:
metrics_df = pd.DataFrame(aggregate_metrics_table(report))
display(metrics_df[["label", "point", "ci_low", "ci_high"]].round(3))

## Performance profile
Main figure for comparing algorithms / reporting in the thesis.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_performance_profile(report, ax=ax)
if COMPARE_REPORT_PATH is not None:
    compare = load_report(COMPARE_REPORT_PATH)
    plot_performance_profile(compare, algorithm=compare["exp_name"], ax=ax, title="Performance profiles")
fig.tight_layout()
plt.show()

## Bootstrap confidence intervals (summary metrics)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
plot_aggregate_intervals(report, ax=ax)
fig.tight_layout()
plt.show()

## Zero-shot generalization by context
Green = training (in-distribution) context; red = OOD physics perturbations.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
plot_generalization_contexts(report, score_matrix, ax=ax)
fig.tight_layout()
plt.show()

## Sample efficiency (optional)
Requires periodic in-distribution eval during training (`eval_history.jsonl`).

In [ ]:
ax = plot_sample_efficiency_curve(report)
if ax is None:
    print("No sample-efficiency curve in report (run longer training with --eval-frequency > 0).")
else:
    ax.figure.tight_layout()
    plt.show()

## Raw score matrix
Rows = seeds, columns = contexts (mean return per eval episodes).

In [ ]:
pd.DataFrame(score_matrix, index=[f"seed {s}" for s in seeds], columns=context_order).round(2)